<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Rules, and when a rule is the right answer

**[Rules, and when a rule is the right answer](https://learning.nextia-ai.com/courses/anomaly-detection/m02/rules-and-when-a-rule-is-the-right-answer/)** · Anomaly Detection: Find What Is Unusual · Module 2, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** write two rules for the shape of a fraud (many payments in a few minutes, one device with many cards), score them in the comparison table, and see which frauds a rule finds and which ones a statistic finds.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | MAD per customer, from [Median, MAD and IQR](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m02/rules-and-when-a-rule-is-the-right-answer/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M02-L03-rules-and-when-a-rule-is-the-right-answer/rules-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The baseline row: no detector

Every comparison starts with "do nothing". Nobody reviews any payment, so every planted fraud is missed. Its cost per week is the price of doing nothing.

> **Check.** You should see a cost per week of 10367 dollars for `no detector`.

In [ ]:
import compare
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
table = compare.Table()
table.add(compare.score_payments(evaluation, np.zeros(len(evaluation)), "no detector", "nothing is reviewed",
                                 threshold=1))
table.show()

## 2. A rule in a tiny example: count the payments in the last 10 minutes

A bank's card team often uses a **velocity rule**: an alert when one card pays too often in a short time. Here the rule is "more than 5 payments in 10 minutes".

The next cell takes the customer of the first lesson on 2025-10-03. For each payment, it counts the payments of the same card in the last 10 minutes, this one included. The pandas method `rolling("600s")` looks back 600 seconds from each row.

> **Predict.** At which of the eight payments does the rule fire for the first time?

In [ ]:
card = evaluation[(evaluation.customer_id == "C-01234") & (evaluation.ts >= "2025-10-03")
                  & (evaluation.ts < "2025-10-04")].set_index("ts")
card["in_last_10_min"] = card.amount.rolling("600s").count().astype(int)   # this payment and the ones before it
card["rule"] = card.in_last_10_min > 5                                     # more than 5 payments in 10 minutes
print(card[["amount", "country", "device_id", "in_last_10_min", "rule"]].to_string())

> **Check.** You should see `in_last_10_min` go from 1 to 8, and `rule` turn `True` at the sixth payment.

The rule is easy to explain: "this card paid 6 times in 10 minutes". But it fires only **after** five payments are done. A rule needs the pattern to happen before it can see it.

## 3. Two rules on all the evaluation payments

Now count for every payment: per card in 10 minutes, and per device in 1 hour. A **device** is the phone or computer that pays. The **device rule** says: an alert when one device makes more than 3 payments in 1 hour. This cell takes a few seconds.

In [ ]:
e = evaluation.sort_values("ts")

def count_in_window(key, window):
    """For each payment: the payments with the same key in the last `window`, this one included."""
    ones = pd.Series(1, index=e.index)
    counts = ones.groupby(e[key]).transform(
        lambda s: s.set_axis(e.loc[s.index, "ts"]).rolling(window).sum().to_numpy())
    return counts.reindex(evaluation.index)

per_card_10min = count_in_window("customer_id", "600s")     # the velocity rule counts per card
per_device_1h = count_in_window("device_id", "3600s")       # the device rule counts per device
print("most payments on one card in 10 minutes:", int(per_card_10min.max()))
print("most payments on one device in 1 hour:", int(per_device_1h.max()))

A rule gives 1 (alert) or 0 (no alert). In the comparison table, a rule's alerts of one day are reviewed in time order, at most 50 a day.

> **Predict.** Which rule will have the higher precision at budget? MAD per customer had 0.048.

In [ ]:
velocity_rule = (per_card_10min > 5).astype(int)
device_rule = (per_device_1h > 3).astype(int)
table.add(compare.score_payments(evaluation, velocity_rule, "velocity rule", "more than 5 payments in 10 minutes",
                                 threshold=1, explain_minutes=0.5))
table.add(compare.score_payments(evaluation, device_rule, "device rule", "a device with more than 3 payments in 1 hour",
                                 threshold=1, explain_minutes=0.5))
table.show()

> **Check.** You should see a precision at budget of 0.133 for the velocity rule and 0.405 for the device rule.

The device rule finds about 8 times more planted frauds per review than MAD per customer. It describes **how this fraud works**: a thief tries many cards on one device.

### What do the rules flag that is not fraud?

In [ ]:
planted = evaluation.payment_id.isin(compare.planted_payments().payment_id)   # the answer key, to score only
for name, rule in (("velocity rule", velocity_rule), ("device rule", device_rule)):
    honest = evaluation[(rule == 1) & ~planted]
    print(f"{name}: {len(honest):,} honest payments flagged, {honest.account_type.eq('business').mean():.1%} business")

Almost every honest payment that the rules flag comes from a business account. A business that buys in bulk pays many times in a few minutes. For the rule, that looks like fraud.

## 4. Rank the alerts: most over the limit first

A rule says only yes or no. A score says **how far** over the limit a payment is, so the team can review the clearest cases first.

In [ ]:
over = np.maximum(per_card_10min - 5, 0) + np.maximum(per_device_1h - 3, 0)   # how far over the limits
table.add(compare.score_payments(evaluation, over, "two rules, ranked", "payments over the limits, most over first",
                                 threshold=0.5, explain_minutes=0.5))
table.show()

> **Check.** You should see a precision at budget of 0.427 for `two rules, ranked`.

## 5. Which frauds does each method find?

The next cell computes MAD per customer again, as in the last lesson, and the recall of each method for each kind of planted fraud.

In [ ]:
g = reference.groupby("customer_id").amount                  # MAD per customer, as in the last lesson
cust = pd.DataFrame({"n": g.size(), "median": g.median(), "mad": g.apply(lambda s: (s - s.median()).abs().median())})
by_type = reference.groupby("account_type").amount.agg(["median", lambda s: (s - s.median()).abs().median()])
by_type.columns = ["median", "mad"]
known = evaluation.customer_id.map(cust["n"]) >= 5
c_median = evaluation.customer_id.map(cust["median"]).where(known).fillna(evaluation.account_type.map(by_type["median"]))
c_mad = np.maximum(evaluation.customer_id.map(cust["mad"]).where(known).fillna(evaluation.account_type.map(by_type["mad"])), 1.0)
rz_customer = ((evaluation.amount - c_median) / (1.4826 * c_mad)).abs()
kinds = pd.DataFrame({
    "MAD per customer": compare.recall_by_kind(evaluation, rz_customer),
    "device rule": compare.recall_by_kind(evaluation, device_rule, threshold=1),
    "velocity rule": compare.recall_by_kind(evaluation, velocity_rule, threshold=1),
})
print(kinds.round(3).to_string())

> **Check.** You should see a recall of 0.534 on `account_takeover` for MAD per customer, and 0.000 for the device rule.

The two methods find **different** frauds. The device rule finds stolen cards and card tests: many payments, one device. It never finds an account takeover: the thief makes one or two large payments from the customer's account. MAD per customer finds those, because the amount is far from the customer's usual one.

## 6. An allow-list, and a rule with a statistic

An **allow-list** is a list of cases that a rule skips on purpose. Here, business accounts skip the device rule, because their bulk orders are the honest alerts that you saw in section 3.

> **Predict.** What happens to the alerts per day and to the precision?

In [ ]:
personal = (evaluation.account_type == "personal").astype(int)     # business accounts are on the allow-list
device_personal = device_rule * personal
row_allow = table.add(compare.score_payments(evaluation, device_personal, "device rule, personal accounts only",
                                             "business accounts on an allow-list", threshold=1, explain_minutes=0.5))
table.show()

> **Check.** You should see 20.4 alerts a day and a precision at budget of 0.981.

Very few false alerts, but the team now has about 30 free review slots each day. Fill them with the highest MAD scores:

In [ ]:
combined = device_personal * 1000 + rz_customer      # rule alerts first, then the highest MAD scores
row_combo = table.add(compare.score_payments(evaluation, combined, "device rule, then MAD per customer",
                                             "rule alerts first, MAD per customer fills the rest", explain_minutes=1))
table.show()

> **Check.** You should see a cost per week of 4,615 dollars for `device rule, then MAD per customer`.

The precision is lower than the rule alone (0.389), because the MAD alerts are less often fraud. But the recall is higher (0.628), and the cost per week is the lowest in the table: the rule and the statistic find different frauds.

**Careful.** The allow-list works so well here because, in this synthetic data, the honest bursts come from business bulk orders. A real shop has more kinds of honest bursts. And a thief with a stolen **business** card now passes the device rule.

## 7. One fraud each method misses

The completion check of this module: find one planted fraud that the device rule finds and MAD per customer misses, and one the other way round. The next cells use the answer key, only to score, as the comparison table does. `True` means "in the reviewed 50 of that day".

In [ ]:
key = compare.planted_payments().set_index("payment_id")
rule_seen = compare.daily_alerts(evaluation, device_rule, threshold=1).set_index("payment_id").reviewed
mad_seen = compare.daily_alerts(evaluation, rz_customer).set_index("payment_id").reviewed
found = pd.DataFrame({"rule": rule_seen, "mad": mad_seen}).loc[key.index.intersection(evaluation.payment_id)]
found["kind"] = key.kind
print(pd.crosstab(found.kind, [found.rule, found.mad]).to_string())

In [ ]:
show = ["payment_id", "ts", "customer_id", "amount", "country", "device_id"]
rule_only = found[found.rule & ~found.mad].index
mad_only = found[found.mad & ~found.rule].index
extra = pd.DataFrame({"payment_id": evaluation.payment_id, "device_1h": per_device_1h, "usual": c_median.round(2),
                      "robust_z": rz_customer.round(1)}).set_index("payment_id")
both = evaluation.set_index("payment_id")[show[1:]].join(extra)
print("found by the device rule, missed by MAD per customer:")
print(both.loc[rule_only[:1]].to_string())
print("found by MAD per customer, missed by the device rule:")
print(both.loc[mad_only[:1]].to_string())

Read the two payments. `device_1h` is the count of the device rule, `usual` is the customer's median, and `robust_z` is the MAD score.

- The first payment is tiny, and its amount is not far from usual (robust z 1.5). But its device paid 4 times in that hour: the pattern that the rule describes.
- The second payment is far above the customer's usual amount (robust z 5.5). But its device count is 3, not more than 3, so the rule stays silent.

## 8. Check yourself

**Your turn 1.** What is the recall of the device rule on `card_testing`? Write it as `card_testing_recall`, rounded to 3 decimals.

In [ ]:
card_testing_recall = ...   # your code

In [ ]:
expect('card_testing_recall', card_testing_recall, 0.522, tolerance=0.0001)

**Your turn 2.** Change one condition. Make the device rule stricter: more than **5** payments in 1 hour. Keep its row as `strict_row`. Does the precision go up or down? And the recall?

In [ ]:
strict_row = ...   # your code: compare.score_payments(evaluation, ..., "device rule, strict", "...", threshold=1)

In [ ]:
expect_hash('the precision and recall of the strict device rule', (strict_row['precision at budget'], strict_row['recall']) if isinstance(strict_row, dict) else None, '0ed87f2522df6109')

**Your turn 3.** How many planted payments are in the reviewed alerts of MAD per customer and **not** in those of the device rule? Use the table `found` of section 7. Write the number as `mad_only_count`.

In [ ]:
mad_only_count = ...   # your code

In [ ]:
expect('mad_only_count', mad_only_count, 156)

## Recap

- A rule describes the shape of a fraud. It is instant and easy to explain: "this device paid 57 times in one hour" is the largest count in these months.
- The device rule found planted frauds with a precision at budget of 0.405; MAD per customer had 0.048. A good rule beats a statistic when it captures how the fraud works.
- A rule misses what nobody wrote down: the device rule never finds an account takeover. MAD per customer does.
- An allow-list removes known honest cases, and opens a door for a thief who looks like them.
- A rule and a statistic together cost less than either one alone.

The next module watches whole processes over time, such as the daily refund rate, with control charts.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Control charts: is this variation normal?](https://learning.nextia-ai.com/courses/anomaly-detection/m03/control-charts/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m02/rules-and-when-a-rule-is-the-right-answer/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.